# Clase 050 — Panorama del ML: tipos, batch vs online, instance vs model-based

Ubicamos cada algoritmo en la grilla `(supervisión × batch/online × instance/model)`.
Comparamos KNN (instance-based) contra LogisticRegression (model-based) y mostramos el
patrón canónico de *online learning* con `partial_fit`.

Requiere: `numpy`, `pandas`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_regression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error

np.random.seed(42)
print('setup ok')

## 1. Supervisado: KNN (instance-based) vs LogReg (model-based)

Ambos resuelven el mismo problema supervisado (clasificar Iris) pero generalizan distinto:
KNN compara por similitud contra los vistos; LogReg aprende parámetros `theta`.

In [ ]:
iris = load_iris()
Xtr, Xte, ytr, yte = train_test_split(iris.data, iris.target, test_size=0.3,
                                      stratify=iris.target, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5).fit(Xtr, ytr)
logreg = LogisticRegression(max_iter=1000).fit(Xtr, ytr)

acc_knn = accuracy_score(yte, knn.predict(Xte))
acc_lr = accuracy_score(yte, logreg.predict(Xte))
print(f'KNN     accuracy test: {acc_knn:.4f}')
print(f'LogReg  accuracy test: {acc_lr:.4f}')

## 2. Instance-based pesa más: evidencia empírica por tamaño serializado

KNN "memoriza" el train set entero; su modelo serializado crece con N. LogReg guarda solo
coeficientes. Verificamos con `pickle.dumps` que el KNN ocupa más bytes.

In [ ]:
bytes_knn = len(pickle.dumps(knn))
bytes_lr = len(pickle.dumps(logreg))
print(f'KNN    serializado: {bytes_knn:5d} bytes')
print(f'LogReg serializado: {bytes_lr:5d} bytes')
assert bytes_knn > bytes_lr, 'KNN deberia pesar mas: guarda el train set'
print('OK: el instance-based ocupa mas porque ES su dataset')

## 3. Batch vs online: `SGDRegressor.partial_fit` en mini-batches

Online learning actualiza el modelo instancia a instancia (o en mini-batches) vía
`partial_fit`. Es el patrón para *streaming* y *out-of-core*. Ploteamos el MSE de train
a medida que llegan los batches.

In [ ]:
X, y = make_regression(n_samples=5000, n_features=10, noise=15.0, random_state=42)
X = StandardScaler().fit_transform(X)
y = (y - y.mean()) / y.std()

sgd = SGDRegressor(learning_rate='invscaling', eta0=0.01, random_state=42)
batch = 500
mse_hist = []
for start in range(0, len(X), batch):
    xb, yb = X[start:start + batch], y[start:start + batch]
    sgd.partial_fit(xb, yb)
    mse_hist.append(mean_squared_error(y, sgd.predict(X)))
print('batches procesados:', len(mse_hist))
print(f'MSE inicial {mse_hist[0]:.4f} -> MSE final {mse_hist[-1]:.4f}')
assert mse_hist[-1] < mse_hist[0], 'el online learning deberia reducir el error'

## 4. Costo de inferencia: instance-based escala con N

KNN compara cada predicción contra todo el train set. Medimos el tiempo de 1000
predicciones para KNN vs LogReg.

In [ ]:
import time
probe = np.repeat(Xte[:1], 1000, axis=0)

t0 = time.perf_counter(); knn.predict(probe); t_knn = time.perf_counter() - t0
t0 = time.perf_counter(); logreg.predict(probe); t_lr = time.perf_counter() - t0
print(f'KNN    1000 preds: {t_knn*1000:.2f} ms')
print(f'LogReg 1000 preds: {t_lr*1000:.2f} ms')
print('KNN es "lazy": mueve el costo de fit a predict')

## 5. La grilla del capítulo 1: ubicar cada algoritmo

Resumimos la taxonomía en una tabla sobre los tres ejes.

In [ ]:
taxonomia = pd.DataFrame([
    ('KNeighborsClassifier', 'supervisado', 'batch',  'instance'),
    ('LogisticRegression',   'supervisado', 'batch',  'model'),
    ('SGDRegressor',         'supervisado', 'online', 'model'),
    ('KMeans',               'no-superv.',  'batch',  'model'),
    ('RandomForest',         'supervisado', 'batch',  'model'),
], columns=['algoritmo', 'supervision', 'batch/online', 'instance/model'])
print(taxonomia.to_string(index=False))

## 6. Visual: tamaño del modelo y velocidad, instance vs model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['KNN\n(instance)', 'LogReg\n(model)'], [bytes_knn, bytes_lr],
            color=['#c33', '#37a'])
axes[0].set_ylabel('bytes serializados')
axes[0].set_title('Instance-based ocupa mas')

axes[1].plot(np.arange(1, len(mse_hist) + 1), mse_hist, marker='o', color='#3a7')
axes[1].set_xlabel('mini-batch #'); axes[1].set_ylabel('MSE en train')
axes[1].set_title('Online learning: el error baja por batch')
plt.tight_layout(); plt.show()

## Ejercicios

1. Agregá `KNeighborsRegressor` y `Ridge` sobre `make_regression` y compará RMSE en test y
   bytes serializados. Confirmá que el patrón instance vs model se repite en regresión.
2. En el loop de `partial_fit`, barajá los datos con `np.random.permutation` antes de armar
   los batches. ¿Cómo cambia la curva de MSE? ¿Por qué el orden importa en online learning?
3. Cambiá `n_neighbors` de KNN a 1 y a 50. ¿Qué le pasa al accuracy y al tiempo de inferencia?
4. Clasificá en la grilla de 3 ejes: `AlphaGo`, un autoencoder y `SGDClassifier`. Justificá
   cada celda (algunos quedan en casos raros).

## Conclusiones

- El marco `(supervisión × batch/online × instance/model)` ubica cualquier algoritmo y
  anticipa su API, forma de evaluarlo y debilidades.
- KNN es instance-based: su "modelo" es el dataset; pesa más y su predicción escala con N.
- `partial_fit` es el patrón canónico de online learning: ideal para streaming y out-of-core.
- No-supervisado no significa "sin entrenar": KMeans igual ajusta centroides con `.fit(X)`.